# all-reduce-eval-metrics — faded example 2: Pack (loss_sum, count) before the all_reduce

> Practice drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `all-reduce-eval-metrics`. The last cell reports your progress on the `Distributed: all_reduce eval metrics` subtopic back to Delta Drills.

**Most of the code is already written — complete the one blanked step**, run the test to check it, then run the last cell to record your progress.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Distributed: all_reduce eval metrics` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`all-reduce-eval-metrics`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "all-reduce-eval-metrics"
DD_SUBTOPIC = "Distributed: all_reduce eval metrics"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

To compute a true sample-weighted global eval loss across ranks with uneven batches, you reduce the *sum of losses* and the *sample count* together, then divide once. The blanked step is building the packed length-2 float tensor that the single `all_reduce(SUM)` will operate on — get the order and the float cast right and the rest follows.

## Faded exercise 2

Below, the `all_reduce` and the final divide are already written. The learner must construct the packed input tensor: a length-2 `float32` tensor whose first element is the local loss sum and whose second element is the local sample count (cast to float). Fill in the `stats = ...` line.

**Your task:** complete the one blanked step in the code cell below. The surrounding code, function signatures, and variable names are given — work out the missing expression yourself, then run the test.

In [ ]:
class MockDist:
    class ReduceOp:
        SUM = 'sum'
    def __init__(self, rank_tensors):
        self.rank_tensors = rank_tensors
    def all_reduce(self, tensor, op=ReduceOp.SUM):
        total = t.zeros_like(self.rank_tensors[0])
        for rt in self.rank_tensors:
            total += rt
        tensor.copy_(total)

def weighted_eval_loss(dist_module, local_loss_sum, local_count):
    stats = None  # TODO: fill in this step — read the prompt cell above
    dist_module.all_reduce(stats, op=dist_module.ReduceOp.SUM)
    return (stats[0] / stats[1]).item()

t.manual_seed(0)
loss_sums = [32.0, 40.0]
counts = [32, 8]
rank_tensors = [t.tensor([ls, float(c)], dtype=t.float32) for ls, c in zip(loss_sums, counts)]
mock = MockDist(rank_tensors)
result = weighted_eval_loss(mock, loss_sums[0], counts[0])
print('weighted global loss:', round(result, 6))


def _test():
    loss_sums = [32.0, 40.0]
    counts = [32, 8]
    expected = sum(loss_sums) / sum(counts)  # 72 / 40 = 1.8
    rank_tensors = [t.tensor([ls, float(c)], dtype=t.float32) for ls, c in zip(loss_sums, counts)]
    mock = MockDist(rank_tensors)
    got = weighted_eval_loss(mock, loss_sums[0], counts[0])
    assert isinstance(got, float), f'expected python float, got {type(got)}'
    assert abs(got - expected) < 1e-6, f'got {got}, expected {expected}'
    assert abs(got - 1.8) < 1e-6, f'got {got}, expected 1.8'


try:
    _test()
    _dd_passed.add('faded2')
    print('[Delta Drills] faded2 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report your progress

Run the cell below to send your progress to Delta Drills. It only counts if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
class MockDist:
    class ReduceOp:
        SUM = 'sum'
    def __init__(self, rank_tensors):
        self.rank_tensors = rank_tensors
    def all_reduce(self, tensor, op=ReduceOp.SUM):
        total = t.zeros_like(self.rank_tensors[0])
        for rt in self.rank_tensors:
            total += rt
        tensor.copy_(total)

def weighted_eval_loss(dist_module, local_loss_sum, local_count):
    stats = t.tensor([float(local_loss_sum), float(local_count)], dtype=t.float32)
    dist_module.all_reduce(stats, op=dist_module.ReduceOp.SUM)
    return (stats[0] / stats[1]).item()

t.manual_seed(0)
loss_sums = [32.0, 40.0]
counts = [32, 8]
rank_tensors = [t.tensor([ls, float(c)], dtype=t.float32) for ls, c in zip(loss_sums, counts)]
mock = MockDist(rank_tensors)
result = weighted_eval_loss(mock, loss_sums[0], counts[0])
print('weighted global loss:', round(result, 6))
```
</details>